%md
# 03 — Schema Standardization

Builds `silver_financial_instrument` (common model) plus the
equity / ETF / fund extension tables that preserve source-specific attributes.

**Dependency:** `dim_exchange` from `04_reference_standardization`.

Run `04_reference_standardization` first.

In [0]:
%run /Workspace/Users/shreyash270204@outlook.com/databricks/utilities/config.py

Config loaded. STORAGE_ACCOUNT=financestorage1 SNAPSHOT_DATE=latest per exchange TAXONOMY_VERSION=v1


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

# Load exchange reference table
dim_exchange = (
    spark.read
    .format("delta")
    .load(silver_path("dim_exchange"))
)

display(dim_exchange)

exchange_code,exchange_name,country_name,iso2,iso3,region,market_type,exchange_key,taxonomy_version,is_current
BER,Berlin Stock Exchange,Germany,DE,DEU,Europe,Equity,4e58ea6a9f51cecf95a87beab90f076472ca1e055af41c48f710c429413b1917,v1,true
BSE,Bombay Stock Exchange,India,IN,IND,Asia,Equity,a44616f05ef4f6545d49ed023eb3c9a3fc4638ddab9c070de94ff615805da786,v1,true
FRA,Frankfurt Stock Exchange,Germany,DE,DEU,Europe,Equity,9a4c1754c2b7c355441c18457c3d0f0852da38b3f568194fbdebb6432e6a0069,v1,true
GER,XETRA,Germany,DE,DEU,Europe,Equity,58d7ea82e6716d7d8712695cb2da50d817effe6355bc70553f0d73141169ae40,v1,true
JPX,Japan Exchange Group,Japan,JP,JPN,Asia,Equity,522b4b317815215f700a53bec0b4731742531b65c921536f6822bb87f84157bf,v1,true
LSE,London Stock Exchange,United Kingdom,GB,GBR,Europe,Equity,b8ec997fafd31d2dfbac31c149e7760117d147fe3369b866a248176a9db17fc3,v1,true
NSE,National Stock Exchange of India,India,IN,IND,Asia,Equity,8bff9e9b043560a5a548ed9bad9f4a7faeaf9258bea37e56ce8e1c09be2d3c0e,v1,true
SHZ,Shenzhen Stock Exchange,China,CN,CHN,Asia,Equity,89718b9178681ecef249bf84a510b789a2773e9ea59925e3fa47fca6981b64dc,v1,true
VIE,Vienna Stock Exchange,Austria,AT,AUT,Europe,Equity,ed72f7ce3d2154ab42711843c6905bfb024c36c2229fc7997100131b61119be1,v1,true
NYQ,New York Stock Exchange,United States,US,USA,North America,Equity,cfab169ca0448be8ef70ee985058a02946db7d4db98653db65e7607e3829bcf5,v1,true


In [0]:
def read_bronze_asset_class(asset_class: str):
    frames = []
    for exch, classes in EXCHANGE_ASSET_COVERAGE.items():
        if asset_class not in classes:
            continue
        path = latest_snapshot_path(asset_class, exch)
        if path is None:
            continue
        df = (
            spark.read.option("header", True).option("inferSchema", True)
            .option("multiLine", True).option("escape", "\"")
            .csv(f"{path}/*.csv")
            .withColumn("exchange", F.coalesce(F.col("exchange"), F.lit(exch)))
            .withColumn("_snapshot_date", F.lit(path.split("snapshot_date=")[-1]))
        )
        frames.append(df)
    if not frames:
        return None
    out = frames[0]
    for f in frames[1:]:
        out = out.unionByName(f, allowMissingColumns=True)
    return out

In [0]:
equities_raw = read_bronze_asset_class("equities")
etfs_raw = read_bronze_asset_class("etfs")
funds_raw = read_bronze_asset_class("funds")

print("Equities:", equities_raw.count() if equities_raw else 0)
print("ETFs:", etfs_raw.count() if etfs_raw else 0)
print("Funds:", funds_raw.count() if funds_raw else 0)

Equities: 40549
ETFs: 13782
Funds: 11774


In [0]:
def dedup_on_symbol_exchange(df, asset_type: str):

    w = (
        Window
        .partitionBy("symbol", "exchange")
        .orderBy(F.lit(1))
    )

    ranked = (
        df
        .withColumn("_rn", F.row_number().over(w))
    )

    deduped = (
        ranked
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    dup_count = (
        ranked
        .filter(F.col("_rn") > 1)
        .count()
    )

    print(
        f"[{asset_type}] "
        f"{dup_count} duplicate symbol+exchange rows dropped"
    )

    return deduped

In [0]:
equities = (
    dedup_on_symbol_exchange(equities_raw, "equities")
    if equities_raw is not None
    else None
)

etfs = (
    dedup_on_symbol_exchange(etfs_raw, "etfs")
    if etfs_raw is not None
    else None
)

funds = (
    dedup_on_symbol_exchange(funds_raw, "funds")
    if funds_raw is not None
    else None
)

[equities] 0 duplicate symbol+exchange rows dropped
[etfs] 0 duplicate symbol+exchange rows dropped
[funds] 0 duplicate symbol+exchange rows dropped


In [0]:
def to_common_model(
    df,
    asset_type: str,
    source_dataset: str,
    has_country: bool,
    has_market: bool
):

    base = (
        df
        .withColumn(
            "asset_type",
            F.lit(asset_type)
        )
        .withColumn(
            "instrument_id",
            F.sha2(
                F.concat_ws(
                    "|",
                    F.lit(asset_type),
                    F.col("symbol"),
                    F.col("exchange")
                ),
                256
            )
        )
        .withColumn(
            "instrument_name",
            F.col("name")
        )
        .withColumn(
            "source_dataset",
            F.lit(source_dataset)
        )
        .withColumn(
            "source_record_id",
            F.concat_ws(
                "|",
                F.col("symbol"),
                F.col("exchange"),
                F.col("_snapshot_date")
            )
        )
    )

    # ETFs and Funds don't have country in bronze.
    # Derive country from dim_exchange.
    if not has_country:

        base = base.join(
            dim_exchange.select(
                F.col("exchange_code").alias("exchange"),
                F.col("country_name").alias("_derived_country")
            ),
            on="exchange",
            how="left"
        )

    country_col = (
        F.col("country")
        if has_country
        else F.col("_derived_country")
    )

    market_col = (
        F.col("market")
        if has_market
        else F.lit(None).cast("string")
    )

    return base.select(
        "instrument_id",
        "symbol",
        "instrument_name",
        "asset_type",
        country_col.alias("country"),
        F.col("currency"),
        F.col("exchange"),
        market_col.alias("market"),
        "source_dataset",
        "source_record_id",
        F.col("_snapshot_date").alias("snapshot_date")
    )

In [0]:
silver_frames = []

if equities is not None:
    silver_frames.append(
        to_common_model(
            equities,
            "equity",
            "equities",
            has_country=True,
            has_market=True
        )
    )

if etfs is not None:
    silver_frames.append(
        to_common_model(
            etfs,
            "etf",
            "etfs",
            has_country=False,
            has_market=False
        )
    )

if funds is not None:
    silver_frames.append(
        to_common_model(
            funds,
            "fund",
            "funds",
            has_country=False,
            has_market=False
        )
    )

silver_financial_instrument = silver_frames[0]

for frame in silver_frames[1:]:
    silver_financial_instrument = (
        silver_financial_instrument
        .unionByName(frame)
    )

display(silver_financial_instrument)

instrument_id,symbol,instrument_name,asset_type,country,currency,exchange,market,source_dataset,source_record_id,snapshot_date
45b3a39e89350593c3035c76207d477cb987f81755e8a998b44e26b884bba3cc,000002.SZ,"China Vanke Co., Ltd.",equity,China,CNY,SHZ,Shenzhen Stock Exchange,equities,000002.SZ|SHZ|2026-08-21,2026-08-21
1e00e7d34122b0149e52c909ab11693b779a06ab31a0ad78ed1607fcdcb63c7b,000004.SZ,"Shenzhen GuoHua Network Security Technology Co., Ltd.",equity,China,CNY,SHZ,Shenzhen Stock Exchange,equities,000004.SZ|SHZ|2026-08-21,2026-08-21
042e5baf0c3a393db805d9d9c268861e68a168508d8089ba8af2a43afbd981e8,000005.SZ,Shenzhen Fountain Corporation,equity,China,CNY,SHZ,Shenzhen Stock Exchange,equities,000005.SZ|SHZ|2026-08-21,2026-08-21
4ed9108f460797b3b88b3dd36f1ddbf6f0f5c7ad6db98583940eb75665c2f0d6,000006.SZ,"Shenzhen Zhenye (Group) Co.,Ltd.",equity,China,CNY,SHZ,Shenzhen Stock Exchange,equities,000006.SZ|SHZ|2026-08-21,2026-08-21
cebd1fe96ae93aad5f21be524718b0d3d467aa684ef1cd569688a45a9642da0d,000007.SZ,"Shenzhen Quanxinhao Co., Ltd.",equity,China,CNY,SHZ,Shenzhen Stock Exchange,equities,000007.SZ|SHZ|2026-08-21,2026-08-21
2e940d8cd1bfd54a908880ed8d6c0c1566ebc583c831d14e98011741627016c9,000009.SZ,"China Baoan Group Co., Ltd.",equity,China,CNY,SHZ,Shenzhen Stock Exchange,equities,000009.SZ|SHZ|2026-08-21,2026-08-21
4232e9755884f732ae43ceeb894ea5d9aac85ca51bcb0b7f944c2ffaecfa0c31,000010.SZ,"Shenzhen Ecobeauty Co., Ltd.",equity,China,CNY,SHZ,Shenzhen Stock Exchange,equities,000010.SZ|SHZ|2026-08-21,2026-08-21
c0ea179016a88b94f576e7d2569d69a71dc2673b1cf210586b9d6a867e5e9cb1,000014.SZ,"Shahe Industrial Co., Ltd",equity,China,CNY,SHZ,Shenzhen Stock Exchange,equities,000014.SZ|SHZ|2026-08-21,2026-08-21
291e41a843f3fbfce1fd1a772c961d31f6b657aa4ab86f0dc3d6a72b8374adf3,000016.SZ,"Konka Group Co., Ltd.",equity,China,CNY,SHZ,Shenzhen Stock Exchange,equities,000016.SZ|SHZ|2026-08-21,2026-08-21
ffe3b9bff83d4737caaaea9e3dc862aea6c9a8ea4041b312c5fff3118f75ea74,000017.SZ,Shenzhen China Bicycle Company (Holdings) Limited,equity,China,CNY,SHZ,Shenzhen Stock Exchange,equities,000017.SZ|SHZ|2026-08-21,2026-08-21


In [0]:
target_path = silver_path("financial_instrument")

if DeltaTable.isDeltaTable(spark, target_path):

    target = DeltaTable.forPath(
        spark,
        target_path
    )

    (
        target.alias("t")
        .merge(
            silver_financial_instrument.alias("s"),
            "t.instrument_id = s.instrument_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:

    (
        silver_financial_instrument.write
        .format("delta")
        .mode("overwrite")
        .save(target_path)
    )

final_count = (
    spark.read
    .format("delta")
    .load(target_path)
    .count()
)

print(
    f"silver_financial_instrument: "
    f"{final_count} total rows after merge"
)

silver_financial_instrument: 66105 total rows after merge


In [0]:
def write_extension(
    df,
    cols,
    table_name,
    asset_type
):

    ext = (
        df
        .withColumn(
            "instrument_id",
            F.sha2(
                F.concat_ws(
                    "|",
                    F.lit(asset_type),
                    F.col("symbol"),
                    F.col("exchange")
                ),
                256
            )
        )
        .select(
            "instrument_id",
            *cols
        )
    )

    path = silver_path(table_name)

    if DeltaTable.isDeltaTable(spark, path):

        target = DeltaTable.forPath(
            spark,
            path
        )

        (
            target.alias("t")
            .merge(
                ext.alias("s"),
                "t.instrument_id = s.instrument_id"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    else:

        (
            ext.write
            .format("delta")
            .mode("overwrite")
            .save(path)
        )

    count = (
        spark.read
        .format("delta")
        .load(path)
        .count()
    )

    print(f"{table_name}: {count} rows")

In [0]:
if equities is not None:

    write_extension(
        equities,
        [
            "sector",
            "industry_group",
            "industry",
            "market_cap"
        ],
        "equity_extension",
        "equity"
    )
    if etfs is not None:

        write_extension(
            etfs,
            [
                "family",
                "category_group",
                "category"
            ],
            "etf_extension",
            "etf"
        )
    if funds is not None:

        write_extension(
            funds,
            [
                "family",
                "category_group",
                "category"
            ],
            "fund_extension",
            "fund"
        )

equity_extension: 40549 rows
etf_extension: 13782 rows
fund_extension: 11774 rows


In [0]:
history = spark.sql(
    f"DESCRIBE HISTORY delta.`{target_path}`"
)

display(
    history.select(
        "version",
        "timestamp",
        "operation"
    )
)

version,timestamp,operation
1,2026-08-21T10:18:33.000Z,MERGE
0,2026-08-21T10:16:17.000Z,WRITE
